In [ ]:
import pandas as pd
import numpy as np

url = "https://www.cpc.ncep.noaa.gov/data/indices/ersst5.nino.mth.91-20.ascii"

raw_df = pd.read_table(
    url,
    sep=r"\s+",
    engine="python"
)

raw_df["YR"] = pd.to_numeric(raw_df["YR"], errors="coerce")
raw_df["MON"] = pd.to_numeric(raw_df["MON"], errors="coerce")
raw_df["NINO3.4"] = pd.to_numeric(raw_df["NINO3.4"], errors="coerce")

clean_df = raw_df[["YR", "MON", "NINO3.4"]].copy()
clean_df = clean_df.replace([-99.99, -99.9, -999, -999.0], np.nan)
clean_df = clean_df.dropna()

clean_df["YR"] = clean_df["YR"].astype(int)
clean_df["MON"] = clean_df["MON"].astype(int)
clean_df["Date"] = pd.to_datetime(dict(year=clean_df["YR"], month=clean_df["MON"], day=1))
clean_df = clean_df.sort_values("Date").reset_index(drop=True)

nino34_series = pd.Series(
    data=clean_df["NINO3.4"].values,
    index=clean_df["Date"],
    name="Nino3.4_Temperature"
)

print(nino34_series.head())
print(nino34_series.tail())
print("Series shape:", nino34_series.shape)

In [ ]:
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

WINDOW_SIZE = 12

values = nino34_series.to_numpy(dtype="float32")

def create_sliding_windows(series_values, window_size):
    X, y = [], []

    for i in range(len(series_values) - window_size):
        X.append(series_values[i:i + window_size])
        y.append(series_values[i + window_size])

    return np.array(X), np.array(y)

X_raw, y_raw = create_sliding_windows(values, WINDOW_SIZE)

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw,
    y_raw,
    train_size=0.80,
    shuffle=False
)

scaler = StandardScaler()
scaler.fit(X_train_raw.reshape(-1, 1))

X_train = scaler.transform(X_train_raw.reshape(-1, 1)).reshape(X_train_raw.shape)
X_test = scaler.transform(X_test_raw.reshape(-1, 1)).reshape(X_test_raw.shape)
y_train = scaler.transform(y_train_raw.reshape(-1, 1)).reshape(-1)
y_test = scaler.transform(y_test_raw.reshape(-1, 1)).reshape(-1)

feature_columns = [f"month_t_minus_{lag}" for lag in range(WINDOW_SIZE, 0, -1)]

X_train_df = pd.DataFrame(X_train, columns=feature_columns)
X_test_df = pd.DataFrame(X_test, columns=feature_columns)
y_train_df = pd.DataFrame({"target_t": y_train})
y_test_df = pd.DataFrame({"target_t": y_test})

candidate_data_dirs = [Path("../data"), Path("data"), Path("ML_FinalProject/data")]
data_dir = next((path for path in candidate_data_dirs if path.exists()), Path("data"))
data_dir.mkdir(parents=True, exist_ok=True)

X_train_df.to_csv(data_dir / "X_train.csv", index=False)
X_test_df.to_csv(data_dir / "X_test.csv", index=False)
y_train_df.to_csv(data_dir / "y_train.csv", index=False)
y_test_df.to_csv(data_dir / "y_test.csv", index=False)

print("X_train shape:", X_train_df.shape)
print("X_test shape:", X_test_df.shape)
print("y_train shape:", y_train_df.shape)
print("y_test shape:", y_test_df.shape)
print("Saved files to:", data_dir.resolve())